# Proyecto dataset de Titanic

Este proyecto tiene por objetivo analizar datos sobre los registros de sobrevivientes del titanic. Utilizando como base de información los archivos que se encuentran en https://www.kaggle.com/competitions/titanic/data. El fin es poder predecir mediante sus datos cuales son las personas que sobreviven o no. Además de identificar cuales son los factores que mas influyen en las probabilidades de sobrevivir.

Algunas explicaciones de columnas:  
- pclass = Clase del ticket: 1st, 2nd, 3rd
- sibsp = Número de hermanos o cónyugues arriba del titanic
- parch = Número de familiares o hijos arriba del titanic
- ticket = Número del ticket
- fare = Tarifa
- cabin = Número de la cabina
- embarked = Puerto de embarcación: C(Cherbourg), Q(Queenstown), S(Southampton)

### Imports necesarios

In [1]:
import pandas as pd
import random
import plotly.express as px

### Carga de datos

In [2]:
ruta_train_csv = r'../data/raw/train.csv'
df_inicial_train = pd.read_csv(ruta_train_csv)

### Visualización inicial

DataFrame cargado inicial:

In [3]:
print(f'Datos iniciales:\n{df_inicial_train}')

Datos iniciales:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                                 .

Info del DataFrame y describe

In [4]:
df_inicial_train.info()
print(f'\nDescribe:\n{df_inicial_train.describe()}')

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

Describe:
       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.83

### EDA

Rellenar valores de Age con la mediana

In [5]:
info_rellenar_datos_age = {'Age': df_inicial_train['Age'].median()}
df_edad_rellenada = df_inicial_train.fillna(info_rellenar_datos_age)
print(f'DataFrame con Age completo:\n{df_edad_rellenada}')
df_edad_rellenada.info()

DataFrame con Age completo:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                       

Quitar la columna Cabin, para este ejercicio no se considerará debido a su cantidad de valores distintos, no se podría clasificar

In [6]:
df_drop_cabin = df_edad_rellenada.drop(columns=['Cabin'])

Rellenar los valores faltantes de Embarked, son pocos pero esto es para tener un DataFrame mas limpio y fácil de trabajar. Se aplica la funcion lambda en donde es "na" se elige un Embarked aleatorio

In [7]:
valores_posibles_embarked = df_drop_cabin['Embarked'].dropna().unique()
print(valores_posibles_embarked)
serie_embarked_procesado = df_drop_cabin['Embarked'].apply(
    lambda x: random.choice(valores_posibles_embarked) if pd.isna(x) else x
)
df_procesado = df_drop_cabin
df_procesado['Embarked'] = serie_embarked_procesado

df_procesado.info()

<StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


### Creación de gráficos para entendimiento

Crear columna de sobrevivientes en palabras, esto para crear un gráfico mas entendible y amigable

In [8]:
df_procesado['Sobreviviente'] = df_procesado['Survived'].map({0: 'No sobrevivió', 1: 'Sobrevivió'})
print(f'DataFrame con columna de sobreviviente o no:\n{df_procesado}')

DataFrame con columna de sobreviviente o no:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                      

Generar DataFrame solo de mujeres, para contar cuantos sobrevivientes y no sobrevivientes hay. Luego mapear colores para que en el gráfico se vean colores especificos dependiendo si es sobreviviente o no.

In [ ]:
df_mujeres = df_procesado[df_procesado['Sex'] == 'female']
cuenta_sobrevivientes_mujeres = df_mujeres['Sobreviviente'].value_counts()
cuenta_sobrevivientes_mujeres = cuenta_sobrevivientes_mujeres.reset_index()
mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

print(cuenta_sobrevivientes_mujeres)

fig_pie_sobrevivientes_female = px.pie(
    cuenta_sobrevivientes_mujeres,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_female.show()


   Sobreviviente  count
0     Sobrevivió    233
1  No sobrevivió     81


Realizar misma accion para los hombres

In [18]:
df_hombres = df_procesado[df_procesado['Sex'] == 'male']
cuenta_sobrevivientes_hombres = df_hombres['Sobreviviente'].value_counts()
cuenta_sobrevivientes_hombres = cuenta_sobrevivientes_hombres.reset_index()
print(cuenta_sobrevivientes_hombres)
mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

fig_pie_sobrevivientes_hombres = px.pie(
    cuenta_sobrevivientes_hombres,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_hombres.show()



   Sobreviviente  count
0  No sobrevivió    468
1     Sobrevivió    109


Ahora buscar taza de sobrevivientes por rangos de edad, primero utilizar una cantidad considerable (mayor a 100) registros por cada grupo para identificar si hay alguna tendencia y que sea fidedigna

In [29]:
df_jovenes_debajo_18 = df_procesado[df_procesado['Age'] <= 18]
df_adultos_18_40 = df_procesado[(df_procesado['Age'] > 18) & (df_procesado['Age'] <= 40)]
df_adultos_sobre_40 = df_procesado[df_procesado['Age'] > 40]
print(len(df_jovenes_debajo_18))
print(len(df_adultos_18_40))
print(len(df_adultos_sobre_40))

139
602
150


Crear y manejar datos para ver sobrevivientes jovenes

In [34]:
sobrevivientes_debajo_18 = df_jovenes_debajo_18['Sobreviviente'].value_counts()
sobrevivientes_debajo_18 = sobrevivientes_debajo_18.reset_index()
print(sobrevivientes_debajo_18)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

fig_pie_sobrevivientes_jovenes = px.pie(
    sobrevivientes_debajo_18,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)
fig_pie_sobrevivientes_jovenes.show()

   Sobreviviente  count
0     Sobrevivió     70
1  No sobrevivió     69


Realizar lo mismo para los pasajeros con edad entre 18 y 40 años

In [37]:
sobrevivientes_18_40 = df_adultos_18_40['Sobreviviente'].value_counts()
sobrevivientes_18_40 = sobrevivientes_18_40.reset_index()
print(sobrevivientes_18_40)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

fig_pie_sobrevivientes_18_40 = px.pie(
    sobrevivientes_18_40,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_18_40.show()

   Sobreviviente  count
0  No sobrevivió    385
1     Sobrevivió    217


Realizar lo mismo para pasajeros mayores a 40 años

In [43]:
sobrevivientes_mayores_40 = df_adultos_sobre_40['Sobreviviente'].value_counts()
sobrevivientes_mayores_40 = sobrevivientes_mayores_40.reset_index()
print(sobrevivientes_mayores_40)

mapeo_colores = {
    'Sobrevivió': '#4365ff',
    'No sobrevivió': '#ff4848'
}

fig_pie_sobrevivientes_mayores_40 = px.pie(
    sobrevivientes_mayores_40,
    values='count',
    names='Sobreviviente',
    color='Sobreviviente',
    color_discrete_map=mapeo_colores
)

fig_pie_sobrevivientes_mayores_40.show()

   Sobreviviente  count
0  No sobrevivió     95
1     Sobrevivió     55
